# RBNO model prediction FEM pointwise residual visualization for `elasticity`

This notebook visualizes the FEM/UFL pointwise residual of the RBNO physics-loss model prediction from checkpoint `best_model_params_3000.pth`.

The RBNO model predicts reduced H(div)xH1 coefficients. These are reconstructed to FEM DoFs, inserted into the mixed `sigma_u` function space, and evaluated with the same UFL residual densities used by `elasticity_gt_fc_residual_diagnostic.ipynb`: `r1` is the constitutive-energy density and `r2` is the equilibrium density.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm

import dolfinx
import scifem
import ufl

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

notebook_cwd = Path.cwd().resolve()
# Launched from FC_PINO/ or from the repo root; derive the repo root either way.
repo_path = notebook_cwd.parent if notebook_cwd.name == 'FC_PINO' else notebook_cwd
fc_pino_path = repo_path / 'FC_PINO'
for path in [str(repo_path), str(fc_pino_path)]:
    while path in sys.path:
        sys.path.remove(path)
for path in [str(fc_pino_path), str(repo_path)]:
    sys.path.insert(0, path)

from data_generation.differential_equations import ElasticityLeastSquares
from utils import evaluate_expression, load_yaml
from models import ConvolutionalNN_65x129

torch.set_default_dtype(torch.float64)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'repo_path: {repo_path}')
print(f'device: {device}')

## Configuration

Defaults visualize one test sample and save the figure under `results/elasticity/model_test_outputs/rbno_physics_loss/visualizations`.

In [ ]:
mesh_config_path = repo_path / 'configs/elasticity/config_data/config_mesh.yaml'
function_space_config_path = repo_path / 'configs/elasticity/config_data/config_function_space.yaml'
output_reduced_basis_config_path = repo_path / 'configs/elasticity/config_data/config_output_reduced_basis.yaml'
train_dataset_path = repo_path / 'results/elasticity/train_dataset'
test_dataset_path = repo_path / 'results/elasticity/test_dataset'
model_train_outputs_path = repo_path / 'results/elasticity/model_train_outputs/rbno_physics_loss'
model_test_outputs_path = repo_path / 'results/elasticity/model_test_outputs/rbno_physics_loss'
visualization_dir = model_test_outputs_path / 'visualizations'
visualization_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = model_train_outputs_path / 'best_model_params_3000.pth'
assert checkpoint_path.exists(), checkpoint_path

dataset_split = 'test'
sample_start_index = 1
num_check = 1
batch_size = 1
num_basis = 512

mesh_args = load_yaml(mesh_config_path)
function_space_args = load_yaml(function_space_config_path)
output_reduced_basis_args = load_yaml(output_reduced_basis_config_path)
num_x = mesh_args['num_x']
num_y = mesh_args['num_y']
Lx = mesh_args['upper_right_x'] - mesh_args['lower_left_x']
Ly = mesh_args['upper_right_y'] - mesh_args['lower_left_y']
print(mesh_args)
print(f'checkpoint: {checkpoint_path}')
print(f'visualization_dir: {visualization_dir}')

## Data loading and RBNO input image

The RBNO checkpoint was trained with parameter input shape `(batch, 1, y_index, x_index)`, using `p_vertex.reshape(num_x + 1, num_y + 1).T`. The FEM residual below is computed entirely from the reconstructed FEM DoFs.

In [ ]:
dataset_path = train_dataset_path if dataset_split == 'train' else test_dataset_path


def load_slice(path: Path, start: int, count: int):
    arr = np.load(path, mmap_mode='r')
    if start < 0 or start >= arr.shape[0]:
        raise IndexError(f'sample_start_index={start} is outside {path.name} with {arr.shape[0]} samples')
    stop = min(start + count, arr.shape[0])
    return np.array(arr[start:stop])

# NOTE: the model input image is NOT built from the on-disk `p_vertex_values.npy`.
# That file is stored in a vertex ordering that does NOT match the perm-scattered
# image ordering the RBNO checkpoint was trained/evaluated with (see the training
# notebook `train/elasticity/elasticity_rbno_physics_loss.ipynb`: it builds
# `p_vertex_values[i][perm] = p.x.array[scifem.vertex_to_dofmap(Vh['CG1'])]`).
# Reshaping the raw file directly scrambles the input and produces a garbage
# prediction with a huge constitutive residual r1. We therefore reconstruct the
# input image from `p_dof` via the same perm-scatter (done in the next FEM cell,
# after the mesh / perm / CG1 space are available).
p_dof = load_slice(dataset_path / 'p_dof.npy', sample_start_index, num_check)
num_check = p_dof.shape[0]

pod_basis_dof_np = np.load(train_dataset_path / 'hdiv_h1_pod_basis_dof.npy')[:, :num_basis]
pod_basis_dof_tensor = torch.as_tensor(pod_basis_dof_np, dtype=torch.float64, device=device)
print('p_dof:', p_dof.shape, 'POD basis:', pod_basis_dof_np.shape)
print('RBNO input image is built from p_dof via perm-scatter in the next FEM cell.')

## Elasticity auxiliary fields

This reproduces the fixed `q` (`z = grad(q)`) and `w` auxiliary fields used in the least-squares residual.

In [ ]:
elasticity_least_squares = ElasticityLeastSquares(mesh_args, function_space_args)
mesh = elasticity_least_squares.mesh
Vh = elasticity_least_squares.Vh

dolfinx_mesh_coords = mesh.geometry.x[:, :2]
x_grid = np.linspace(mesh_args['lower_left_x'], mesh_args['upper_right_x'], num_x + 1)
y_grid = np.linspace(mesh_args['lower_left_y'], mesh_args['upper_right_y'], num_y + 1)
image_mesh_coords = np.array(np.meshgrid(x_grid, y_grid)).T.reshape(-1, 2)
perm = np.array([np.where(np.isclose(image_mesh_coords, row).all(axis=1))[0][0] for row in dolfinx_mesh_coords], dtype=np.int64)
max_perm_coordinate_mismatch = np.max(np.abs(image_mesh_coords[perm] - dolfinx_mesh_coords))
print(f'max mesh/image coordinate mismatch after perm: {max_perm_coordinate_mismatch:.2e}')
assert max_perm_coordinate_mismatch < 1e-12

q_fc = elasticity_least_squares.solve_q()
w_fc = elasticity_least_squares.solve_w()


def mesh_scalar_values_to_xy(values_at_mesh):
    values_at_mesh = np.asarray(values_at_mesh).reshape(-1)
    flat = np.zeros(len(perm), dtype=np.float64)
    flat[perm] = values_at_mesh
    return flat.reshape(num_x + 1, num_y + 1)


# Build the RBNO input image EXACTLY as the training notebook does:
# interpolate p_dof to CG1, scatter vertex values into image-grid order via `perm`,
# then reshape with `.reshape(num_x + 1, num_y + 1).T`. This matches the convention
# the checkpoint was trained with and reproduces the trained model's predictions.
vertex_to_cg1_dof = scifem.vertex_to_dofmap(Vh['CG1'])
num_vertices = dolfinx_mesh_coords.shape[0]
rbno_p_image = np.zeros((num_check, 1, num_y + 1, num_x + 1), dtype=np.float32)
for i in range(num_check):
    p_cg1 = dolfinx.fem.Function(Vh['CG1'])
    p_cg1.x.array[:] = p_dof[i]
    p_vertex_image_order = np.zeros(num_vertices, dtype=np.float64)
    p_vertex_image_order[perm] = p_cg1.x.array[:][vertex_to_cg1_dof]
    rbno_p_image[i, 0] = p_vertex_image_order.reshape(num_x + 1, num_y + 1).T
rbno_input_tensor = torch.as_tensor(rbno_p_image, dtype=torch.float32, device=device)

print('q, w auxiliary fields solved.')
print('RBNO input (perm-scatter from p_dof):', tuple(rbno_input_tensor.shape))

## RBNO checkpoint inference and DoF reconstruction

In [ ]:
model = ConvolutionalNN_65x129(output_dim=num_basis, activation='leakyrelu', init_func='xavier_uniform')
state = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(state['model_state_dict'] if isinstance(state, dict) and 'model_state_dict' in state else state)
model = model.to(device)
model.eval()
model_dtype = next(model.parameters()).dtype
rbno_model_input_tensor = rbno_input_tensor.to(dtype=model_dtype)
print(f'RBNO model parameter dtype: {model_dtype}; inference input dtype: {rbno_model_input_tensor.dtype}')

with torch.no_grad():
    rbno_coeff_pred = model(rbno_model_input_tensor).to(torch.float64)
    rbno_pred_dof_tensor = rbno_coeff_pred @ pod_basis_dof_tensor.T

rbno_pred_dof = rbno_pred_dof_tensor.detach().cpu().numpy().astype(np.float64)
np.save(model_test_outputs_path / 'visualized_rbno_prediction_sigma_u_dof.npy', rbno_pred_dof)
print('RBNO coeff:', tuple(rbno_coeff_pred.shape), 'prediction dof:', rbno_pred_dof.shape)

## FEM/UFL residual on the RBNO prediction

This follows the FEM pointwise method from `elasticity_gt_fc_residual_diagnostic.ipynb`: form the residual densities from the predicted FEM functions and evaluate them at mesh vertices.

In [ ]:
def averaged_rectangle_rule_np(values):
    values = np.asarray(values)
    x_res, y_res = values.shape[-2], values.shape[-1]
    number_of_rectangle_cells = x_res * y_res
    rectangle_cell_volume = (Lx * Ly) / number_of_rectangle_cells
    sampled_domain_volume = rectangle_cell_volume * number_of_rectangle_cells
    rectangle_integral = np.sum(values, axis=(-2, -1)) * rectangle_cell_volume
    return rectangle_integral / sampled_domain_volume


def pointwise_fem_residual_arrays(sample_index):
    p_fc = dolfinx.fem.Function(Vh['p'])
    p_fc.x.array[:] = p_dof[sample_index]

    sigma_u_fc = dolfinx.fem.Function(Vh['sigma_u'])
    sigma_u_fc.x.array[:] = rbno_pred_dof[sample_index]
    sigma_fc = sigma_u_fc.sub(0).collapse()
    sigma1_fc, sigma2_fc = ufl.split(sigma_fc)
    sigma_tensor = ufl.as_vector((sigma1_fc, sigma2_fc))
    u_fc = sigma_u_fc.sub(1).collapse()

    lambda_, mu = elasticity_least_squares.get_lame_parameters(p=p_fc)

    def C_inv_action(s):
        return 1.0 / (2.0 * mu) * s - lambda_ / (4.0 * mu * (lambda_ + mu)) * ufl.tr(s) * ufl.Identity(2)

    def C_action(e):
        return 2.0 * mu * e + lambda_ * ufl.tr(e) * ufl.Identity(2)

    z = ufl.grad(q_fc)
    epsilon = ufl.sym(ufl.grad(u_fc + w_fc))
    T = sigma_tensor + z - C_action(epsilon)
    r1_expr = ufl.inner(T, C_inv_action(T))
    div_sigma = ufl.as_vector([ufl.div(sigma1_fc), ufl.div(sigma2_fc)])
    r2_expr = ufl.inner(div_sigma, div_sigma)

    r1_vals = evaluate_expression(mesh, r1_expr, mesh.geometry.x)[1][:, 0]
    r2_vals = evaluate_expression(mesh, r2_expr, mesh.geometry.x)[1][:, 0]
    r1_xy = mesh_scalar_values_to_xy(r1_vals)
    r2_xy = mesh_scalar_values_to_xy(r2_vals)
    return r1_xy, r2_xy

fem_residual_loss_dict = {
    'loss_1': np.zeros(num_check),
    'loss_2': np.zeros(num_check),
    'total_loss': np.zeros(num_check),
    'sqrt_total_loss': np.zeros(num_check),
}
for i in tqdm(range(num_check), desc='RBNO prediction FEM pointwise residual'):
    fem_r1, fem_r2 = pointwise_fem_residual_arrays(i)
    fem_r1_avg = averaged_rectangle_rule_np(fem_r1)
    fem_r2_avg = averaged_rectangle_rule_np(fem_r2)
    total = fem_r1_avg + fem_r2_avg
    fem_residual_loss_dict['loss_1'][i] = fem_r1_avg
    fem_residual_loss_dict['loss_2'][i] = fem_r2_avg
    fem_residual_loss_dict['total_loss'][i] = total
    fem_residual_loss_dict['sqrt_total_loss'][i] = np.sqrt(total)

np.save(model_test_outputs_path / 'visualized_rbno_prediction_fem_pointwise_residual_loss_dict.npy', fem_residual_loss_dict)
print(f"mean RBNO prediction FEM pointwise residual loss 1: {np.mean(fem_residual_loss_dict['loss_1']):.2e} (std: {np.std(fem_residual_loss_dict['loss_1']):.2e})")
print(f"mean RBNO prediction FEM pointwise residual loss 2: {np.mean(fem_residual_loss_dict['loss_2']):.2e} (std: {np.std(fem_residual_loss_dict['loss_2']):.2e})")
print(f"mean RBNO prediction FEM pointwise total residual: {np.mean(fem_residual_loss_dict['total_loss']):.2e} (std: {np.std(fem_residual_loss_dict['total_loss']):.2e})")

## RBNO FEM residual visualization

The figure uses the original pointwise residual density. Each panel uses a grayscale colorbar with limits set by the 1% and 99% quantiles of that FEM pointwise residual image.

In [ ]:
extent = [mesh_args['lower_left_x'], mesh_args['upper_right_x'], mesh_args['lower_left_y'], mesh_args['upper_right_y']]
num_visualize_residual = num_check


def quantile_limits(values, lower=0.01, upper=0.99):
    finite_values = np.asarray(values)[np.isfinite(values)]
    if finite_values.size == 0:
        return 0.0, 1.0
    vmin, vmax = np.quantile(finite_values, [lower, upper])
    if not np.isfinite(vmin) or not np.isfinite(vmax):
        return float(np.nanmin(finite_values)), float(np.nanmax(finite_values))
    if np.isclose(vmin, vmax):
        delta = max(abs(float(vmin)) * 1e-6, 1e-12)
        vmin -= delta
        vmax += delta
    return float(vmin), float(vmax)


def plot_rbno_fem_residual_pair(fem_r1, fem_r2, local_sample_index):
    global_sample_index = sample_start_index + local_sample_index
    residuals = [
        (fem_r1, r'RBNO residual $r_1^2$'),
        (fem_r2, r'RBNO residual $r_2^2$'),
    ]

    fig, axs = plt.subplots(2, 1, figsize=(11, 11), constrained_layout=False)
    fig.subplots_adjust(left=0.04, right=0.9, bottom=0.06, top=0.92, hspace=0.18)

    for ax, (values, title_base) in zip(axs, residuals):
        mean_value = float(averaged_rectangle_rule_np(values))
        vmin, vmax = quantile_limits(values)
        im = ax.imshow(values.T, extent=extent, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
        ax.set_title(f'{title_base}\n(mean={mean_value:.2e})', fontsize=22, pad=12)
        ax.set_aspect('equal', adjustable='box')
        ax.set_xticks([])
        ax.set_yticks([])
        cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
        cbar.locator = ticker.MaxNLocator(nbins=4)
        formatter = ticker.ScalarFormatter(useMathText=False)
        formatter.set_scientific(True)
        formatter.set_powerlimits((0, 0))
        cbar.formatter = formatter
        cbar.update_ticks()
        cbar.ax.tick_params(labelsize=18)
        cbar.ax.yaxis.get_offset_text().set_fontsize(18)

    path = visualization_dir / f'sample_{global_sample_index}_rbno_fem_residual_r1_r2.png'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return path

pointwise_residual_visualization_files = []
pointwise_residual_visualization_stats = []
for local_sample_index in tqdm(range(num_visualize_residual), desc='RBNO FEM residual visualizations'):
    fem_r1, fem_r2 = pointwise_fem_residual_arrays(local_sample_index)

    pointwise_residual_visualization_files.append(str(plot_rbno_fem_residual_pair(fem_r1, fem_r2, local_sample_index)))

    pointwise_residual_visualization_stats.append({
        'sample_index': sample_start_index + local_sample_index,
        'fem_r1_rectangle_average': float(averaged_rectangle_rule_np(fem_r1)),
        'fem_r2_rectangle_average': float(averaged_rectangle_rule_np(fem_r2)),
        'fem_total_rectangle_average': float(averaged_rectangle_rule_np(fem_r1 + fem_r2)),
        'fem_r1_max': float(np.max(fem_r1)),
        'fem_r2_max': float(np.max(fem_r2)),
        'fem_total_max': float(np.max(fem_r1 + fem_r2)),
    })

print(f'Saved {len(pointwise_residual_visualization_files)} RBNO FEM residual figure(s) to {visualization_dir}')
print(json.dumps(pointwise_residual_visualization_stats, indent=2))

## Save summary

In [ ]:
def summarize_loss_dict(loss_dict):
    return {key + '_mean': float(np.mean(value)) for key, value in loss_dict.items()} | {key + '_std': float(np.std(value)) for key, value in loss_dict.items()}

metrics_summary = {
    'dataset_split': dataset_split,
    'sample_start_index': sample_start_index,
    'num_check': num_check,
    'checkpoint_path': str(checkpoint_path),
    'num_basis': num_basis,
    'residual_method': 'FEM/UFL pointwise residual densities evaluated at mesh vertices, matching elasticity_gt_fc_residual_diagnostic.ipynb',
    'model_architecture': {
        'model': 'ConvolutionalNN_65x129',
        'input_shape': [1, num_y + 1, num_x + 1],
        'input_convention': 'p_vertex.reshape(num_x + 1, num_y + 1).T, matching RBNO training notebook',
        'output_dim': num_basis,
        'reconstruction': 'predicted reduced coefficients @ hdiv_h1_pod_basis_dof.T',
    },
    'pointwise_grid_average': 'averaged rectangle rule over vertex-evaluated pointwise residual densities',
    'rbno_prediction_fem_pointwise_residual': summarize_loss_dict(fem_residual_loss_dict),
    'pointwise_residual_visualization_files': pointwise_residual_visualization_files,
    'pointwise_residual_visualization_stats': pointwise_residual_visualization_stats,
}

summary_path = visualization_dir / 'rbno_prediction_fem_pointwise_residual_visualization_summary.json'
with open(summary_path, 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print(json.dumps(metrics_summary, indent=2))
print(f'saved summary to {summary_path}')